In [2]:
import langchain
import chromadb
import pypdf
import os

DATA_PATH = r"C:\Users\Dhyey\Documents\Python\Kaggle\Datasets\RAG\documents"

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader(r"C:\Users\Dhyey\Documents\Python\Kaggle\Datasets\RAG\documents\Monster_Hunter.pdf")
documents = pdf_loader.load()
print(documents)

C:\Users\Dhyey\AppData\Local\Temp\ipykernel_5492\3409710051.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-07-19T13:47:00+00:00', 'title': 'Monster Hunter - Wikipedia', 'moddate': '2026-07-19T13:47:00+00:00', 'source': 'C:\\Users\\Dhyey\\Documents\\Python\\Kaggle\\Datasets\\RAG\\documents\\Monster_Hunter.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}, page_content="Monster Hunter\nLogo for Monster Hunter\nGenre Action role-playing\nDeveloper Capcom\nPublisher Capcom\nCreator Kaname Fujioka[1]\nPlatforms PlayStation 2, PlayStation 3,\nPlayStation 4, PlayStation 5,\nPlayStation Portable,\nPlayStation Vita, Windows,\nWii, Wii U, Xbox 360, Xbox\nOne, Xbox Series X/S,\nNintendo 3DS, Nintendo\nSwitch, Nintendo Switch 2,\nAndroid, iOS\nFirst release Monster Hunter\nMarch 11, 2004\nLatest release Monster Hunter Wilds\nFebruary 28, 2025\nMonster Hunter\nMonster Hunter ( モンスターハンター, Monsutā\nHantā) is 

In [4]:
documents[19].metadata

{'producer': 'Skia/PDF m149',
 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/149.0.0.0 Safari/537.36',
 'creationdate': '2026-07-19T13:47:00+00:00',
 'title': 'Monster Hunter - Wikipedia',
 'moddate': '2026-07-19T13:47:00+00:00',
 'source': 'C:\\Users\\Dhyey\\Documents\\Python\\Kaggle\\Datasets\\RAG\\documents\\Monster_Hunter.pdf',
 'total_pages': 20,
 'page': 19,
 'page_label': '20'}

In [5]:
len(documents)

20

In [6]:
char_count = 0
for page in range(len(documents)):
    char_count += len(documents[page].page_content)

# documents[19].page_content
print(char_count)

53605


In [7]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)

chunks = splitter.split_documents(documents)

In [25]:
# print(chunks)
print([len(chunk.page_content) for chunk in chunks])

[468, 487, 488, 454, 468, 472, 458, 415, 203, 397, 451, 448, 425, 417, 426, 495, 400, 493, 394, 411, 402, 311, 420, 431, 413, 407, 397, 399, 359, 476, 494, 410, 490, 406, 420, 306, 463, 450, 489, 411, 497, 429, 409, 421, 432, 413, 414, 418, 404, 300, 500, 479, 496, 421, 361, 491, 453, 399, 461, 446, 435, 379, 457, 483, 417, 486, 474, 419, 478, 493, 453, 478, 495, 368, 454, 479, 491, 499, 466, 469, 246, 487, 428, 487, 407, 409, 406, 396, 419, 403, 477, 399, 398, 455, 462, 410, 448, 450, 383, 444, 446, 441, 445, 474, 454, 475, 459, 482, 478, 492, 461, 481, 475, 454, 457, 415, 412, 432, 416, 488, 423, 427, 435, 470, 425, 489, 470, 481, 484, 453, 490, 444, 494, 483, 434, 473, 410, 440, 455, 435, 431, 474, 484, 427, 492, 344, 464, 475, 497, 438, 424, 428, 490, 450, 464, 454, 492, 445, 418, 435, 465, 424, 479, 431, 428, 484, 468, 468, 422, 450, 494, 425, 235, 421, 424, 468, 423, 269]


In [ ]:
from langchain_chroma import Chroma
from langchain_voyageai import VoyageAIEmbeddings

embedding_model = VoyageAIEmbeddings(model="voyage-4-lite")

CHROMA_DIR = "../chroma_db"

if os.path.exists(CHROMA_DIR):
    vector_store = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embedding_model
    )

else:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=CHROMA_DIR
    )

In [27]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

In [29]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following pieces of context to answer the question at the end.
If you don't know the answer, say that you don't know.
Context: {context}
Question: {question}
""")

In [30]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [31]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("How many mainline Monster Hunter games are there and what years did they come out?")
print(result)

Based on the provided context, there are **6** primary games in the **Main series** (mainline) of *Monster Hunter*. 

Their original release years are:

1. **Monster Hunter** – 2004 (March 11, 2004)
2. **Monster Hunter 2** – 2006 (February 16, 2006)
3. **Monster Hunter Tri** – 2009 (August 1, 2009)
4. **Monster Hunter 4** – 2013 (September 14, 2013)
5. **Monster Hunter: World** – 2018 (January 26, 2018)
6. **Monster Hunter Wilds** – 2025 (Scheduled for February 28, 2025)

*(Note: The context also lists several enhanced versions, expansions, and portable entries, but these 6 are the primary titles categorized under the "Main series" table).*
